TASK 1

In [1]:
!pip install PyPDF2

In [4]:
import PyPDF2
import re
from collections import Counter

#Part (a): Extract text from each page
pdf_file = input("Enter PDF file path: ")
#C:\Users\DELL\Downloads\AI Article.pdf

pages = []

with open(pdf_file, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    for page in reader.pages:
        text = page.extract_text()
        pages.append(text if text else "")

print("Total pages extracted:", len(pages))


#Part (b): Merge pages and create tokens
all_text = " ".join(pages)

#remove punctuation and digits
clean_text = re.sub(r"[^a-zA-Z\s]", "", all_text)

#convert to lowercase
clean_text = clean_text.lower()

#split text into words
tokens = clean_text.split()

print("Total token count:", len(tokens))


#Part (c): Word frequency analysis
stopwords = {
    "the", "is", "a", "an", "and", "or", "of", "to", "in",
    "on", "for", "with", "at", "by", "from", "as", "it",
    "this", "that", "are", "was", "were", "be", "been", "has",
    "have", "had", "but", "not"
}

#remove stopwords
filtered_words = []

for word in tokens:
    if word not in stopwords:
        filtered_words.append(word)

#count frequency of each word
word_frequency = Counter(filtered_words)

print("\nTop 10 most frequent words:")

for word, count in word_frequency.most_common(10):
    print(word, ":", count)


#Part (d): Search for a keyword
keyword = input("\nEnter a keyword to search: ").lower()

found = False

for page_number, page_text in enumerate(pages, start=1):

    #split page text into sentences
    sentences = re.split(r'(?<=[.!?])\s+', page_text)

    for sentence in sentences:
        if keyword in sentence.lower():
            print("\nPage", page_number, ":", sentence.strip())
            found = True

if not found:
    print("No sentence containing the keyword was found.")

Total pages extracted: 5
Total token count: 149

Top 10 most frequent words:
intelligence : 6
artificial : 5
systems : 5
learning : 3
language : 3
use : 3
ai : 3
computer : 2
science : 2
can : 2

Page 2 : Machine
 
learning
 
is
 
an
 
important
 
part
 
of
 
artificial
 
intelligence.

Page 2 : Machine
 
learning
 
algorithms
 
can
 
identify
 
patterns
 
in
 
large
 
datasets
 
and
 
use
 
these
 
patterns
 
to
 
make
 
predictions.


TASK 2

In [ ]:
import PyPDF2
import csv


#Part (a): Extract page information
pdf_file = input("Enter PDF file path: ").strip('"')

pages_data = []

with open(pdf_file, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    for page_number, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text()

        if raw_text is None:
            raw_text = ""

        #count words
        words = raw_text.split()
        word_count = len(words)

        #count characters excluding spaces
        char_count = len(raw_text.replace(" ", "").replace("\n", ""))

        page_data = {
            "page_number": page_number,
            "raw_text": raw_text,
            "word_count": word_count,
            "char_count": char_count
        }

        pages_data.append(page_data)


print("First 2 entries:")
print(pages_data[:2])


#Part (b): Find highest, lowest and average word count

#remove pages with zero words
non_zero_pages = [page for page in pages_data if page["word_count"] > 0]

if len(non_zero_pages) > 0:

    #page with highest word count
    highest_page = max(non_zero_pages, key=lambda page: page["word_count"])

    print("\nPage with highest word count:")
    print("Page number:", highest_page["page_number"])
    print("Word count:", highest_page["word_count"])
    print("Content:")
    print(highest_page["raw_text"])

    #page with lowest non-zero word count
    lowest_page = min(non_zero_pages, key=lambda page: page["word_count"])

    print("\nPage with lowest non-zero word count:")
    print("Page number:", lowest_page["page_number"])
    print("Word count:", lowest_page["word_count"])

    #average word count
    total_words = sum(page["word_count"] for page in pages_data)
    average_words = total_words / len(pages_data)

    print("\nAverage word count:", round(average_words, 2))

else:
    print("All pages contain zero words.")


#Part (c): Search for a phrase

class PhraseNotFoundError(Exception):
    pass


phrase = input("\nEnter a search phrase: ").lower()

try:
    found = False

    for page in pages_data:
        text = page["raw_text"].lower()

        count = text.count(phrase)

        if count > 0:
            print("Page", page["page_number"], "-", count, "occurrence(s)")
            found = True

    if not found:
        raise PhraseNotFoundError

except PhraseNotFoundError:
    print("PhraseNotFoundError: The phrase was not found in any page.")


#Part (d): Write page summary to CSV

with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as file:

    writer = csv.writer(file)

    #write column headings
    writer.writerow(["page_number", "word_count", "char_count"])

    for page in pages_data:
        writer.writerow([
            page["page_number"],
            page["word_count"],
            page["char_count"]
        ])


#read the CSV file again
print("\nContents of pdf_summary.csv:")

with open("pdf_summary.csv", "r", encoding="utf-8") as file:

    reader = csv.reader(file)

    for row in reader:
        print(row)

TASK 3

In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download("stopwords")

In [ ]:
import PyPDF2
import re
import nltk
from nltk.corpus import stopwords


#Part (a): Extract and clean PDF text

pdf_file = input("Enter PDF file path: ").strip('"')

all_text = ""

with open(pdf_file, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    for page in reader.pages:
        text = page.extract_text()

        if text:
            all_text += text + " "

print("Character count before cleaning:", len(all_text))


#remove URLs
cleaned_text = re.sub(r"https?://\S+|www\.\S+", "", all_text)

#remove digits
cleaned_text = re.sub(r"\d+", "", cleaned_text)

#remove non-alphabetic characters except spaces and full stops
cleaned_text = re.sub(r"[^a-zA-Z\s.]", "", cleaned_text)

#collapse multiple spaces
cleaned_text = re.sub(r"\s+", " ", cleaned_text)

print("Character count after cleaning:", len(cleaned_text))


#Part (b): Convert to lowercase and split into sentences

cleaned_text = cleaned_text.lower()

sentences = re.split(r"[.!?]", cleaned_text)

print("\nSentence count before removing empty strings:", len(sentences))

#remove leading/trailing spaces
sentences = [sentence.strip() for sentence in sentences]

#remove empty sentences
sentences = [sentence for sentence in sentences if sentence]

print("Sentence count after removing empty strings:", len(sentences))


#Part (c): Tokenize and remove stopwords

stop_words = set(stopwords.words("english"))

filtered_sentences = []

for sentence in sentences:

    words = sentence.split()

    #remove stopwords
    filtered_words = [
        word for word in words
        if word not in stop_words
    ]

    #keep sentence only if 5 or more words remain
    if len(filtered_words) >= 5:
        filtered_sentence = " ".join(filtered_words)
        filtered_sentences.append(filtered_sentence)


print("\nSentence count before token filter:", len(sentences))
print("Sentence count after token filter:", len(filtered_sentences))


#display 3 sample sentences
print("\n3 Sample sentences:")

for sentence in filtered_sentences[:3]:
    print(sentence)


#Part (d): Write remaining sentences to a file

with open("nlp_ready_output.txt", "w", encoding="utf-8") as file:

    for sentence in filtered_sentences:
        file.write(sentence + "\n")


#re-read the file
with open("nlp_ready_output.txt", "r", encoding="utf-8") as file:

    output_sentences = [line.strip() for line in file if line.strip()]


print("\nTotal lines written:", len(output_sentences))


if len(output_sentences) > 0:

    #find longest sentence
    longest_sentence = max(output_sentences, key=len)

    #find shortest sentence
    shortest_sentence = min(output_sentences, key=len)

    print("\nLongest sentence:")
    print(longest_sentence)
    print("Character length:", len(longest_sentence))

    print("\nShortest sentence:")
    print(shortest_sentence)
    print("Character length:", len(shortest_sentence))

else:
    print("No sentences were left after filtering.")

TASK 4

In [ ]:
!pip install python-docx

In [ ]:
from docx import Document
from statistics import mean
from pprint import pprint


#Part (a): Read the DOCX file

docx_file = input("Enter DOCX file path: ").strip('"') 
#"C:\Users\DELL\Downloads\docx for task 4.docx"

document = Document(docx_file)

paragraphs_data = []

for paragraph in document.paragraphs:

    paragraph_data = {
        "text": paragraph.text,
        "style": paragraph.style.name
    }

    paragraphs_data.append(paragraph_data)


print("Total paragraph count:", len(paragraphs_data))

#find all unique styles
unique_styles = set()

for paragraph in paragraphs_data:
    unique_styles.add(paragraph["style"])

print("\nUnique style names:")
for style in unique_styles:
    print(style)


#Part (b): Classify paragraphs

headings = []
body = []
other_empty = []

for paragraph in paragraphs_data:

    text = paragraph["text"]
    style = paragraph["style"]

    if style.startswith("Heading"):
        headings.append(paragraph)

    elif style == "Normal" or "Body" in style:
        body.append(paragraph)

    else:
        other_empty.append(paragraph)


print("\nParagraph classification:")
print("Headings:", len(headings))
print("Body:", len(body))
print("Other/Empty:", len(other_empty))


#Part (c): Statistical analysis of Body paragraphs

qualifying_body = []

for paragraph in body:

    words = paragraph["text"].split()

    if len(words) > 5:

        word_count = len(words)

        #remove spaces from the text
        char_count = len(
            paragraph["text"].replace(" ", "").replace("\n", "")
        )

        qualifying_body.append({
            "text": paragraph["text"],
            "word_count": word_count,
            "char_count": char_count
        })


if len(qualifying_body) > 0:

    #paragraph with highest word count
    highest = max(
        qualifying_body,
        key=lambda paragraph: paragraph["word_count"]
    )

    print("\nParagraph with highest word count:")
    print(highest["text"])
    print("Word count:", highest["word_count"])
    print("Character count:", highest["char_count"])


    #paragraph with lowest word count above 5
    lowest = min(
        qualifying_body,
        key=lambda paragraph: paragraph["word_count"]
    )

    print("\nParagraph with lowest word count above 5:")
    print(lowest["text"])
    print("Word count:", lowest["word_count"])
    print("Character count:", lowest["char_count"])


    #mean word count
    word_counts = [
        paragraph["word_count"]
        for paragraph in qualifying_body
    ]

    mean_word_count = mean(word_counts)

    print("\nMean word count:", round(mean_word_count, 2))

else:
    print("\nNo Body paragraphs have more than 5 words.")


#Part (d): Create nested dictionary for all styles

style_data = {}

for paragraph in paragraphs_data:

    style = paragraph["style"]
    text = paragraph["text"]

    if style not in style_data:
        style_data[style] = {
            "count": 0,
            "texts": []
        }

    style_data[style]["count"] += 1
    style_data[style]["texts"].append(text)


#show only first 2 texts if a style has more than 2 paragraphs
for style in style_data:

    if style_data[style]["count"] > 2:
        style_data[style]["texts"] = style_data[style]["texts"][:2]


#sort styles by count in descending order
sorted_style_data = dict(
    sorted(
        style_data.items(),
        key=lambda item: item[1]["count"],
        reverse=True
    )
)


print("\nNested style dictionary:")
pprint(sorted_style_data)

TASK 5

In [ ]:
from docx import Document
from docx.oxml.ns import qn
import pandas as pd


#Part (a): Extract all tables

docx_file = input("Enter DOCX file path: ").strip('"')
#"C:\Users\DELL\Downloads\docx for task 5.docx"

doc = Document(docx_file)

tables_data = []

for table in doc.tables:

    table_data = []

    for row in table.rows:

        row_data = []

        for cell in row.cells:
            row_data.append(cell.text.strip())

        table_data.append(row_data)

    tables_data.append(table_data)


print("Total number of tables:", len(tables_data))

for i, table in enumerate(tables_data, start=1):

    rows = len(table)

    if rows > 0:
        columns = len(table[0])
    else:
        columns = 0

    print("Table", i, "dimensions:", rows, "x", columns)


#Part (b): Create DataFrame from first table

if len(tables_data) > 0:

    first_table = tables_data[0]

    #first row is the header
    headers = first_table[0]

    #remaining rows are the data
    data = first_table[1:]

    df = pd.DataFrame(data, columns=headers)

    print("\nFirst table as DataFrame:")
    print(df)


    #count empty or None cells in each column
    print("\nEmpty or None cells per column:")

    for column in df.columns:

        empty_count = df[column].isna().sum()

        #also count empty strings
        empty_count += (df[column] == "").sum()

        print(column, ":", empty_count)

else:
    print("No tables found in the document.")


#Part (c): Extract paragraphs before the first table

pre_table_paragraphs = []

body = doc.element.body

for element in body:

    #stop when the first table is reached
    if element.tag == qn("w:tbl"):
        break

    #check if the element is a paragraph
    if element.tag == qn("w:p"):

        text = "".join(
            node.text or ""
            for node in element.iter()
            if node.tag == qn("w:t")
        )

        pre_table_paragraphs.append(text.strip())


print("\nNumber of pre-table paragraphs:", len(pre_table_paragraphs))


#Part (d): Write extracted data to a text file

with open("docx_extracted.txt", "w", encoding="utf-8") as file:

    file.write("--- PRE-TABLE PARAGRAPHS ---\n")

    for paragraph in pre_table_paragraphs:
        file.write(paragraph + "\n")


    file.write("\n--- TABLE DATA ---\n")

    if len(tables_data) > 0:

        for row in tables_data[0]:

            file.write(" | ".join(row) + "\n")


print("\nData successfully written to docx_extracted.txt")

TASK 6

In [ ]:
from docx import Document
import re
from collections import Counter


#Part (a): Extract headings from both documents

file_a = input("Enter Document A path: ").strip('"')
#"C:\Users\DELL\Downloads\doc A.docx"
file_b = input("Enter Document B path: ").strip('"')
#"C:\Users\DELL\Downloads\doc B.docx"

doc_a = Document(file_a)

doc_b = Document(file_b)


headings_a = []
headings_b = []

for paragraph in doc_a.paragraphs:

    if paragraph.style.name.startswith("Heading"):
        headings_a.append(paragraph.text.strip())


for paragraph in doc_b.paragraphs:

    if paragraph.style.name.startswith("Heading"):
        headings_b.append(paragraph.text.strip())


#convert lists to sets for comparison
set_a = set(headings_a)
set_b = set(headings_b)


print("\nHeadings common to both documents:")
print(set_a & set_b)

print("\nHeadings only in Document A:")
print(set_a - set_b)

print("\nHeadings only in Document B:")
print(set_b - set_a)


#Part (b): Extract Normal paragraphs

normal_a = []
normal_b = []

for paragraph in doc_a.paragraphs:

    if paragraph.style.name == "Normal":
        normal_a.append(paragraph.text)


for paragraph in doc_b.paragraphs:

    if paragraph.style.name == "Normal":
        normal_b.append(paragraph.text)


#merge both lists
all_body_paragraphs = normal_a + normal_b

print("\nTotal body paragraphs before deduplication:",
      len(all_body_paragraphs))


#remove exact duplicates
unique_paragraphs = []
seen = set()

for paragraph in all_body_paragraphs:

    cleaned_paragraph = paragraph.strip().lower()

    if cleaned_paragraph not in seen:

        seen.add(cleaned_paragraph)
        unique_paragraphs.append(paragraph.strip())


print("Total body paragraphs after deduplication:",
      len(unique_paragraphs))


#Part (c): Word frequency analysis

stopwords = {
    "the", "is", "a", "an", "and", "or",
    "of", "to", "in", "on", "for", "with",
    "at", "by", "from", "as", "it"
}


all_words = []

for paragraph in unique_paragraphs:

    #remove punctuation
    cleaned_text = re.sub(r"[^a-zA-Z\s]", "", paragraph)

    #convert to lowercase
    cleaned_text = cleaned_text.lower()

    #split into words
    words = cleaned_text.split()

    #remove stopwords
    for word in words:

        if word not in stopwords:
            all_words.append(word)


#count word frequencies
word_frequency = Counter(all_words)


print("\nTop 20 most frequent words:")

for word, count in word_frequency.most_common(20):
    print(word, ":", count)


#Part (d): Summary

total_headings = len(headings_a) + len(headings_b)

total_unique_paragraphs = len(unique_paragraphs)

vocabulary_size = len(word_frequency)


#calculate paragraph lengths
paragraph_lengths = []

for paragraph in unique_paragraphs:

    cleaned_text = re.sub(r"[^a-zA-Z\s]", "", paragraph)

    words = cleaned_text.split()

    paragraph_lengths.append(len(words))


if len(paragraph_lengths) > 0:

    average_paragraph_length = (
        sum(paragraph_lengths) / len(paragraph_lengths)
    )

    longest_index = paragraph_lengths.index(
        max(paragraph_lengths)
    )

    longest_paragraph = unique_paragraphs[longest_index]

else:

    average_paragraph_length = 0
    longest_paragraph = ""


print("\n========== SUMMARY ==========")

print("Total headings across both documents:",
      total_headings)

print("Total unique body paragraphs:",
      total_unique_paragraphs)

print("Total unique word types:",
      vocabulary_size)

print("Average paragraph length:",
      round(average_paragraph_length, 2),
      "words")

print("\nLongest paragraph:")
print(longest_paragraph)

print("Word count:", max(paragraph_lengths, default=0))

TASK 7

In [ ]:
!pip install requests

In [ ]:
import requests


#Part (a): Get data from both APIs

users_url = "https://jsonplaceholder.typicode.com/users"
posts_url = "https://jsonplaceholder.typicode.com/posts"


users_response = requests.get(users_url)
posts_response = requests.get(posts_url)


#check status codes
if users_response.status_code != 200:
    raise ConnectionError(
        f"Users API returned status code {users_response.status_code}"
    )

if posts_response.status_code != 200:
    raise ConnectionError(
        f"Posts API returned status code {posts_response.status_code}"
    )


#convert responses to JSON
users = users_response.json()
posts = posts_response.json()


print("Users API status code:", users_response.status_code)
print("Posts API status code:", posts_response.status_code)

print("Number of users:", len(users))
print("Number of posts:", len(posts))


#Part (b): Join users and posts

joined_data = {}


for user in users:

    user_id = user["id"]
    user_name = user["name"]

    #find all posts belonging to this user
    user_posts = []

    for post in posts:

        if post["userId"] == user_id:
            user_posts.append(post["title"])


    joined_data[user_name] = {
        "email": user["email"],
        "post_count": len(user_posts),
        "titles": user_posts,
        "company": user["company"]["name"]
    }


#display first 3 users
print("\nFirst 3 users:")

for name, data in list(joined_data.items())[:3]:

    print("\nName:", name)
    print("Email:", data["email"])
    print("Post count:", data["post_count"])
    print("Titles:", data["titles"])


#Part (c): Find the user with the most posts

most_posts_user = max(
    joined_data.items(),
    key=lambda item: item[1]["post_count"]
)


user_name = most_posts_user[0]
user_data = most_posts_user[1]


print("\n========== USER WITH MOST POSTS ==========")

print("Full name:", user_name)
print("Email:", user_data["email"])
print("Company:", user_data["company"])

print("Post titles:")

for number, title in enumerate(user_data["titles"], start=1):
    print(number, ".", title)


#Part (d): Filter users

print("\n========== MATCHING USERS ==========")

for user in users:

    company_bs = user["company"]["bs"].lower()
    city = user["address"]["city"]

    #check both conditions
    condition_1 = "synergies" in company_bs
    condition_2 = len(city) > 8

    if condition_1 or condition_2:

        name = user["name"]

        post_count = joined_data[name]["post_count"]

        print("\nName:", name)
        print("City:", city)
        print("Post count:", post_count)

TASK 8

In [ ]:
import requests
import sys


#Part (a): Fetch and validate todo data

todos_url = "https://jsonplaceholder.typicode.com/todos"

response = requests.get(todos_url)

if response.status_code != 200:
    print("Error: Todo API returned status code", response.status_code)
    sys.exit()

todos = response.json()

if not isinstance(todos, list):
    print("Error: The API response is not a list.")
    sys.exit()

if len(todos) == 0:
    print("Error: The todo list is empty.")
    sys.exit()

print("Total todo records retrieved:", len(todos))


#Part (b): Group todos by userId

user_todos = {}

for todo in todos:

    user_id = todo["userId"]

    if user_id not in user_todos:
        user_todos[user_id] = {
            "total": 0,
            "completed": 0,
            "incomplete": 0
        }

    user_todos[user_id]["total"] += 1

    if todo["completed"]:
        user_todos[user_id]["completed"] += 1
    else:
        user_todos[user_id]["incomplete"] += 1


print("\nTodo summary for each user:")

for user_id in sorted(user_todos):

    data = user_todos[user_id]

    print(
        "User", user_id,
        ":", data
    )

    check = (
        data["completed"] + data["incomplete"]
        == data["total"]
    )

    print("Completed + Incomplete == Total:", check)


#Part (c): Calculate completion rates

completion_rates = {}

for user_id, data in user_todos.items():

    rate = (
        data["completed"] / data["total"]
    ) * 100

    completion_rates[user_id] = round(rate, 1)


#highest completion rate
highest_user = max(
    completion_rates,
    key=completion_rates.get
)

highest_rate = completion_rates[highest_user]


#lowest completion rate
lowest_user = min(
    completion_rates,
    key=completion_rates.get
)

lowest_rate = completion_rates[lowest_user]


#overall average
average_rate = (
    sum(completion_rates.values())
    / len(completion_rates)
)


print("\nCompletion Rate Analysis:")

print(
    "Highest completion rate: User",
    highest_user,
    "-", highest_rate, "%"
)

print(
    "Lowest completion rate: User",
    lowest_user,
    "-", lowest_rate, "%"
)

print(
    "Overall average completion rate:",
    round(average_rate, 1),
    "%"
)


#Part (d): Fetch users and create lookup dictionary

users_url = "https://jsonplaceholder.typicode.com/users"

users_response = requests.get(users_url)

if users_response.status_code != 200:
    print(
        "Error: Users API returned status code",
        users_response.status_code
    )
    sys.exit()

users = users_response.json()


user_lookup = {}

for user in users:
    user_lookup[user["id"]] = user["name"]


#sort users by completion rate in descending order
sorted_users = sorted(
    user_todos.items(),
    key=lambda item: completion_rates[item[0]],
    reverse=True
)


#Print formatted table

print("\n")
print(
    f"{'Rank':<6}"
    f"{'Name':<25}"
    f"{'Total Tasks':<15}"
    f"{'Completed':<12}"
    f"{'Completion Rate %':<20}"
)

print("-" * 78)


for rank, (user_id, data) in enumerate(
    sorted_users,
    start=1
):

    name = user_lookup.get(
        user_id,
        "Unknown"
    )

    rate = completion_rates[user_id]

    print(
        f"{rank:<6}"
        f"{name:<25}"
        f"{data['total']:<15}"
        f"{data['completed']:<12}"
        f"{rate:<20.1f}"
    )

TASK 9

In [ ]:
import requests
import pandas as pd
import math


#Part (a): Fetch and flatten user data

users_url = "https://jsonplaceholder.typicode.com/users"

response = requests.get(users_url)

if response.status_code != 200:
    raise ConnectionError(
        f"Users API returned status code {response.status_code}"
    )

users = response.json()


def flatten_user(user: dict) -> dict:

    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"]
    }


flattened_users = []

for user in users:
    flattened_users.append(flatten_user(user))


print("Flattened user data:")
for user in flattened_users:
    print(user)


#Part (b): Create DataFrame and calculate distance

df = pd.DataFrame(flattened_users)

print("\nFull DataFrame:")
print(df.to_string(index=False))


#convert latitude and longitude to float
df["lat"] = df["lat"].astype(float)
df["lng"] = df["lng"].astype(float)


#calculate distance from origin
def calculate_distance(row):

    return math.sqrt(
        row["lat"] ** 2 +
        row["lng"] ** 2
    )


df["distance_from_origin"] = df.apply(
    calculate_distance,
    axis=1
)


#find user with greatest distance
farthest_user = df.loc[
    df["distance_from_origin"].idxmax()
]


print("\nUser with greatest distance from origin:")

print("Name:", farthest_user["name"])
print("Distance:", farthest_user["distance_from_origin"])


#Part (c): Classify users by hemisphere

northern_users = df[df["lat"] > 0]
southern_users = df[df["lat"] < 0]


print("\nNorthern Hemisphere users:")

for name in northern_users["name"]:
    print(name)


print("\nSouthern Hemisphere users:")

for name in southern_users["name"]:
    print(name)


#calculate average distance for each hemisphere
northern_average = northern_users[
    "distance_from_origin"
].mean()

southern_average = southern_users[
    "distance_from_origin"
].mean()


print("\nAverage distance - Northern Hemisphere:",
      round(northern_average, 2))

print("Average distance - Southern Hemisphere:",
      round(southern_average, 2))


if northern_average > southern_average:
    print("Northern Hemisphere has the higher average distance.")

elif southern_average > northern_average:
    print("Southern Hemisphere has the higher average distance.")

else:
    print("Both hemispheres have the same average distance.")


#Part (d): Fetch posts and comments

posts_url = "https://jsonplaceholder.typicode.com/posts"
comments_url = "https://jsonplaceholder.typicode.com/comments"


posts_response = requests.get(posts_url)
comments_response = requests.get(comments_url)


if posts_response.status_code != 200:
    raise ConnectionError(
        f"Posts API returned status code {posts_response.status_code}"
    )

if comments_response.status_code != 200:
    raise ConnectionError(
        f"Comments API returned status code {comments_response.status_code}"
    )


posts = posts_response.json()
comments = comments_response.json()


#count comments for each post
comment_counts = {}

for post in posts:
    comment_counts[post["id"]] = 0


for comment in comments:

    post_id = comment["postId"]

    if post_id in comment_counts:
        comment_counts[post_id] += 1


#build list of tuples
post_data = []

for post in posts:

    post_id = post["id"]
    title = post["title"][:40]
    comment_count = comment_counts[post_id]

    post_data.append(
        (post_id, title, comment_count)
    )


#sort by comment count in descending order
post_data.sort(
    key=lambda item: item[2],
    reverse=True
)


#top 5 most-commented posts
print("\n========== TOP 5 MOST-COMMENTED POSTS ==========")

for post in post_data[:5]:
    print(
        "Post ID:", post[0],
        "| Title:", post[1],
        "| Comments:", post[2]
    )


#bottom 5 least-commented posts
print("\n========== BOTTOM 5 LEAST-COMMENTED POSTS ==========")

for post in post_data[-5:]:
    print(
        "Post ID:", post[0],
        "| Title:", post[1],
        "| Comments:", post[2]
    )